# Bài 6 – Trang 27: Phân lớp Mushroom

**6451071011 - Lê Nguyễn Anh Dự**

Chạy các ô theo thứ tự trong Google Colab. Notebook tự tải `mushrooms.csv` từ liên kết Drive nếu tệp chưa có trong thư mục làm việc.


In [ ]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

STUDENT = '6451071011 - Lê Nguyễn Anh Dự'
FILE_ID = '11OXFpnwUwK_Ka9BFmboMoT9RhTRe3CFW'

def heading(number, title):
    print(STUDENT)
    print(f'Câu {number}. {title}')

def ensure_data():
    filename = 'mushrooms.csv'
    if not os.path.exists(filename):
        try:
            import gdown
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'gdown'])
            import gdown
        gdown.download(id=FILE_ID, output=filename, quiet=False)
    if not os.path.exists(filename):
        raise FileNotFoundError('Không tải được mushrooms.csv; hãy tải tệp lên Colab.')
    return filename

data = pd.read_csv(ensure_data(), skipinitialspace=True)
data.columns = data.columns.str.strip()
for col in data.select_dtypes(include=['object']).columns:
    data[col] = data[col].str.strip()
assert 'class' in data.columns, 'Tệp phải có cột class.'
assert set(data['class'].dropna().unique()).issubset({'e', 'p'}), 'Nhãn class phải là e hoặc p.'
print('Dữ liệu:', data.shape)
display(data.head())


## Câu 1. Tách thuộc tính đầu vào và nhãn quyết định

In [ ]:
heading(1, 'Tách X và y')
X = data.drop(columns='class').copy()
y = data['class'].copy()
print('X:', X.shape, '| y:', y.shape)
print('Phân bố nhãn (e: ăn được; p: độc):')
display(y.value_counts())


## Câu 2. Mã hóa one-hot các cột không phải số

Ký hiệu `?` trong dữ liệu nấm là một mức giá trị chưa biết của thuộc tính, được giữ như một nhóm riêng khi one-hot để không tự ý bỏ mẫu.


In [ ]:
heading(2, 'One-hot encoding')
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
print('Số cột cần mã hóa:', len(categorical_columns))
print('Tên cột:', categorical_columns)
X = pd.get_dummies(X, columns=categorical_columns, dtype=np.uint8)
print('Kích thước sau mã hóa:', X.shape)
display(X.head())


## Câu 3. Chia dữ liệu huấn luyện 70% và kiểm thử 30%

In [ ]:
heading(3, 'Chia train/test')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f'Huấn luyện: {len(X_train)} mẫu ({len(X_train)/len(X):.1%})')
print(f'Kiểm thử: {len(X_test)} mẫu ({len(X_test)/len(X):.1%})')


## Hàm đánh giá dùng chung

`DecisionTreeClassifier(criterion="entropy")` dùng information gain theo hướng dẫn. Đây là cách mô phỏng ID3 bằng scikit-learn, không phải cài đặt ID3 thuần túy.


In [ ]:
class_names = ['e', 'p']
results = []

def evaluate(name, model):
    pred = model.predict(X_test)
    print(classification_report(y_test, pred, labels=class_names,
                                target_names=['Ăn được (e)', 'Độc (p)'], zero_division=0))
    cm = confusion_matrix(y_test, pred, labels=class_names)
    print('Ma trận nhầm lẫn (hàng: thực tế; cột: dự đoán):')
    print(cm)
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Ăn được (e)', 'Độc (p)'],
                yticklabels=['Ăn được (e)', 'Độc (p)'])
    plt.xlabel('Dự đoán'); plt.ylabel('Thực tế'); plt.title(name)
    plt.tight_layout(); plt.show()
    results.append({'Mô hình': name,
                    'Accuracy': accuracy_score(y_test, pred),
                    'Precision (độc)': precision_score(y_test, pred, pos_label='p', zero_division=0),
                    'Recall (độc)': recall_score(y_test, pred, pos_label='p', zero_division=0),
                    'F1 (độc)': f1_score(y_test, pred, pos_label='p', zero_division=0)})

def show_tree(model, title, filename):
    print('Bốn tầng đầu của cây; mô hình được huấn luyện trên cây đầy đủ:')
    print(export_text(model, feature_names=list(X_train.columns), max_depth=3))
    plt.figure(figsize=(22, 10))
    plot_tree(model, feature_names=list(X_train.columns),
              class_names=['Ăn được', 'Độc'], filled=True, rounded=True,
              max_depth=3, fontsize=8)
    plt.title(title + ' – 4 tầng đầu')
    plt.tight_layout()
    plt.savefig(filename, dpi=100, bbox_inches='tight')
    plt.show()
    print('Đã lưu hình:', filename)


## Câu 4. Cây ID3 theo entropy

In [ ]:
heading(4, 'Cây ID3/entropy')
id3 = DecisionTreeClassifier(criterion='entropy', random_state=42)
id3.fit(X_train, y_train)
evaluate('ID3 (entropy)', id3)
show_tree(id3, 'ID3 (entropy)', 'mushroom_tree_id3.png')


## Câu 5. Cây CART theo Gini

In [ ]:
heading(5, 'Cây CART/Gini')
cart = DecisionTreeClassifier(criterion='gini', random_state=42)
cart.fit(X_train, y_train)
evaluate('CART (gini)', cart)
show_tree(cart, 'CART (gini)', 'mushroom_tree_cart.png')


## Câu 6. Naïve Bayes

In [ ]:
heading(6, 'Gaussian Naïve Bayes')
nb = GaussianNB()
nb.fit(X_train, y_train)
evaluate('Gaussian Naïve Bayes', nb)


## Câu 7. So sánh kết quả

In [ ]:
heading(7, 'So sánh ba mô hình')
comparison = pd.DataFrame(results).set_index('Mô hình')
display(comparison.style.format('{:.4f}').highlight_max(axis=0, color='#c8e6c9'))
for metric in ['Accuracy', 'Recall (độc)', 'F1 (độc)']:
    best = comparison[metric].idxmax()
    print(f'{metric} cao nhất: {best} ({comparison.loc[best, metric]:.4f})')
print('Nhận xét: Recall cho lớp nấm độc cho biết tỷ lệ nấm độc được phát hiện; nếu bỏ sót nấm độc, chúng sẽ bị dự đoán là ăn được. F1 cân bằng precision và recall, còn accuracy phản ánh tỷ lệ đúng chung. So sánh cần dựa trên các số liệu thực tế ở bảng và ma trận nhầm lẫn phía trên.')
